In [1]:
"""
Exploratory Data Analysis - Diabetic Patient Readmission Data
Dataset: UCI Diabetes 130-US Hospitals (1999–2008)
~101,766 encounters × 50 features
"""

# =============================================================
# 0. SETUP
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

# -----------------------------------------------------------
# NOTE: update this path to match your repo layout
BASE = "https://raw.githubusercontent.com/wip-0/ds207_final_project/main/data/interim/split_groupshuffle/"
# -----------------------------------------------------------

X_train = pd.read_csv(BASE + "X_train_mini.csv", na_values="?", low_memory=False)
y_train = pd.read_csv(BASE + "y_train_mini.csv")

# Rejoin for EDA
df = pd.concat([X_train, y_train], axis=1)
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")


Dataset shape: (101766, 50)
Rows: 101,766  |  Columns: 50


In [2]:

# =============================================================
# 1. COLUMN OVERVIEW
# =============================================================
print("\n=== COLUMN OVERVIEW ===")
print(df.dtypes.rename("dtype").to_frame()
      .assign(nulls=df.isnull().sum(),
              pct_null=(df.isnull().mean() * 100).round(1))
      .to_string())




=== COLUMN OVERVIEW ===
                          dtype  nulls  pct_null
encounter_id              int64      0       0.0
patient_nbr               int64      0       0.0
race                        str   2273       2.2
gender                      str      0       0.0
age                         str      0       0.0
weight                      str  98569      96.9
admission_type_id         int64      0       0.0
discharge_disposition_id  int64      0       0.0
admission_source_id       int64      0       0.0
time_in_hospital          int64      0       0.0
payer_code                  str  40256      39.6
medical_specialty           str  49949      49.1
num_lab_procedures        int64      0       0.0
num_procedures            int64      0       0.0
num_medications           int64      0       0.0
number_outpatient         int64      0       0.0
number_emergency          int64      0       0.0
number_inpatient          int64      0       0.0
diag_1                      str     21      

In [3]:

# =============================================================
# 2. MISSING VALUES
# =============================================================
miss = df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(miss) * 0.4)))
miss.mul(100).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("% Missing")
ax.set_title("Missing Value Rate by Column", fontweight="bold")
ax.axvline(50, color="tomato", linestyle="--", linewidth=1.2, label="50 % threshold")
ax.legend()
plt.tight_layout()
plt.savefig("fig_01_missing_values.png")
plt.close()
print("Saved: fig_01_missing_values.png")

# Columns with >50 % missing – likely to drop
high_miss = miss[miss > 0.5].index.tolist()
print(f"\nColumns >50% missing (likely to drop): {high_miss}")

Saved: fig_01_missing_values.png

Columns >50% missing (likely to drop): ['weight', 'max_glu_serum', 'A1Cresult']


In [4]:


# =============================================================
# 3. TARGET VARIABLE: readmitted
# =============================================================
target_counts = df["readmitted"].value_counts()
target_pct    = df["readmitted"].value_counts(normalize=True).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# count bar
target_counts.plot(kind="bar", ax=axes[0], color=["#4878d0","#ee854a","#6acc64"],
                   edgecolor="white", width=0.6)
axes[0].set_title("Readmission Counts", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_xticklabels(target_counts.index, rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=10)

# pie
axes[1].pie(target_counts, labels=target_counts.index,
            autopct="%1.1f%%", startangle=140,
            colors=["#4878d0","#ee854a","#6acc64"])
axes[1].set_title("Readmission Distribution", fontweight="bold")

plt.suptitle("Target: readmitted  (NO / <30 days / >30 days)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_02_target_distribution.png")
plt.close()
print("Saved: fig_02_target_distribution.png")


Saved: fig_02_target_distribution.png


In [5]:


# =============================================================
# 4. PATIENT DEMOGRAPHICS
# =============================================================

# ---- 4a. Age distribution ----
age_order = ["[0-10)","[10-20)","[20-30)","[30-40)","[40-50)",
             "[50-60)","[60-70)","[70-80)","[80-90)","[90-100)"]
age_ct = df["age"].value_counts().reindex(age_order)

fig, ax = plt.subplots(figsize=(9, 4))
age_ct.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white", width=0.7)
ax.set_title("Patient Age Distribution", fontweight="bold")
ax.set_xlabel("Age group")
ax.set_ylabel("Count")
ax.set_xticklabels(age_order, rotation=45, ha="right")
plt.tight_layout()
plt.savefig("fig_03_age_distribution.png")
plt.close()
print("Saved: fig_03_age_distribution.png")

# ---- 4b. Race & Gender ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

race_ct = df["race"].value_counts(dropna=False)
race_ct.plot(kind="bar", ax=axes[0], color="teal", edgecolor="white", width=0.6)
axes[0].set_title("Race", fontweight="bold")
axes[0].set_xticklabels(race_ct.index, rotation=30, ha="right")

gen_ct = df["gender"].value_counts()
gen_ct.plot(kind="bar", ax=axes[1], color=["#ee854a","#4878d0","gray"], edgecolor="white", width=0.5)
axes[1].set_title("Gender", fontweight="bold")
axes[1].set_xticklabels(gen_ct.index, rotation=0)

plt.suptitle("Demographics", fontweight="bold")
plt.tight_layout()
plt.savefig("fig_04_demographics.png")
plt.close()
print("Saved: fig_04_demographics.png")



Saved: fig_03_age_distribution.png
Saved: fig_04_demographics.png


In [6]:

# =============================================================
# 5. NUMERIC FEATURE DISTRIBUTIONS
# =============================================================
num_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures",
            "num_medications", "number_outpatient", "number_emergency",
            "number_inpatient", "number_diagnoses"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color="steelblue", edgecolor="white", linewidth=0.4)
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("")
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color="tomato", linestyle="--", linewidth=1.2,
                    label=f"mean={mean_val:.1f}")
    axes[i].legend(fontsize=8)

plt.suptitle("Numeric Feature Distributions", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_05_numeric_distributions.png")
plt.close()
print("Saved: fig_05_numeric_distributions.png")



Saved: fig_05_numeric_distributions.png


In [7]:
# =============================================================
# 6. NUMERIC FEATURES vs READMISSION
# =============================================================
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
read_order = ["NO", ">30", "<30"]
palette = {"NO": "#4878d0", ">30": "#ee854a", "<30": "#6acc64"}

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x="readmitted", y=col, order=read_order,
                palette=palette, ax=axes[i], flierprops={"marker":".", "markersize":2})
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("")

plt.suptitle("Numeric Features by Readmission Status", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_06_numeric_vs_readmission.png")
plt.close()
print("Saved: fig_06_numeric_vs_readmission.png")

Saved: fig_06_numeric_vs_readmission.png


In [8]:

# =============================================================
# 7. MEDICATION USAGE OVERVIEW
# =============================================================
med_cols = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride",
            "glipizide","glyburide","pioglitazone","rosiglitazone","insulin"]

# Count non-"No" prescriptions
def active(series):
    return (series != "No").mean() * 100

med_usage = pd.Series({c: active(df[c]) for c in med_cols}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
med_usage.plot(kind="bar", ax=ax, color="mediumseagreen", edgecolor="white", width=0.6)
ax.set_title("Medication Usage Rate (% encounters with non-'No' value)", fontweight="bold")
ax.set_ylabel("% of encounters")
ax.set_xticklabels(med_usage.index, rotation=45, ha="right")
plt.tight_layout()
plt.savefig("fig_07_medication_usage.png")
plt.close()
print("Saved: fig_07_medication_usage.png")


Saved: fig_07_medication_usage.png


In [9]:


# =============================================================
# 8. A1C RESULT & GLUCOSE SERUM vs READMISSION
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["A1Cresult", "max_glu_serum"]):
    ct = df.groupby(col)["readmitted"].value_counts(normalize=True).unstack().fillna(0) * 100
    ct.plot(kind="bar", ax=ax, edgecolor="white", width=0.6,
            color=["#4878d0","#ee854a","#6acc64"])
    ax.set_title(f"{col} vs Readmission (%)", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("% of group")
    ax.set_xticklabels(ct.index, rotation=30, ha="right")
    ax.legend(title="readmitted", fontsize=8)

plt.tight_layout()
plt.savefig("fig_08_a1c_glucose_vs_readmission.png")
plt.close()
print("Saved: fig_08_a1c_glucose_vs_readmission.png")


Saved: fig_08_a1c_glucose_vs_readmission.png


In [10]:

# =============================================================
# 9. CORRELATION HEATMAP (numeric columns)
# =============================================================
corr_df = df[num_cols + ["admission_type_id", "discharge_disposition_id",
                         "admission_source_id"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation Matrix – Numeric Features", fontweight="bold")
plt.tight_layout()
plt.savefig("fig_09_correlation_heatmap.png")
plt.close()
print("Saved: fig_09_correlation_heatmap.png")



Saved: fig_09_correlation_heatmap.png


In [11]:

# =============================================================
# 10. READMISSION RATE BY AGE GROUP
# =============================================================
readmit_age = (df.groupby("age")["readmitted"]
               .apply(lambda x: (x != "NO").mean() * 100)
               .reindex(age_order))

fig, ax = plt.subplots(figsize=(9, 4))
readmit_age.plot(kind="bar", ax=ax, color="coral", edgecolor="white", width=0.6)
ax.set_title("Readmission Rate by Age Group (any readmission)", fontweight="bold")
ax.set_ylabel("% readmitted")
ax.set_xticklabels(age_order, rotation=45, ha="right")
plt.tight_layout()
plt.savefig("fig_10_readmission_by_age.png")
plt.close()
print("Saved: fig_10_readmission_by_age.png")

Saved: fig_10_readmission_by_age.png


In [12]:

# =============================================================
# SUMMARY STATS EXPORT
# =============================================================
summary = df.describe(include="all").T
summary.to_csv("summary_statistics.csv")
print("\nSaved: summary_statistics.csv")

print("\n  EDA complete — all figures and summary_statistics.csv written.")



Saved: summary_statistics.csv

  EDA complete — all figures and summary_statistics.csv written.
